# Boltz-2 co-folding check (AChE, 4EY7 chain A)
GPU = T4/P100, Internet = On. MODE below: `smoke` runs one molecule (donepezil) to test the whole chain; `full` runs everything. Every step aborts the notebook on error.

In [ ]:
MODE = "full"   # 'smoke' or 'full'
import os, sys, shutil, subprocess, glob
def sh(cmd, **kw):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, executable='/bin/bash', **kw)
    if r.returncode != 0:
        raise SystemExit('FAILED (exit %d): %s' % (r.returncode, cmd))

In [ ]:
# 1. isolated Python 3.11 environment (Boltz-2's dependency dm-tree has no wheel for Kaggle's default Python 3.13)
sh('pip -q install uv')
sh('uv venv --python 3.11 /tmp/venv')
sh('uv pip install --python /tmp/venv/bin/python boltz pandas scikit-learn biopython gemmi')
sh('''/tmp/venv/bin/python -c "import torch, boltz; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0))"''')
sh('/tmp/venv/bin/boltz predict --help > /dev/null')

In [ ]:
# 2. copy the inputs next to the scripts (everything heavy lives in /tmp so the notebook output stays small)
base = None
for r, d, f in os.walk('/kaggle/input'):
    if 'boltz_run.py' in f:
        base = os.path.dirname(r); break
assert base, 'input dataset not found'
root = '/tmp/pkg'; os.makedirs(root, exist_ok=True)
for d in ('data', 'docking', 'scripts'):
    shutil.copytree(os.path.join(base, d), os.path.join(root, d), dirs_exist_ok=True)
os.chdir(root)
import time
os.environ.update(BOLTZ_T0=str(time.time()), BOLTZ_COPY='/kaggle/working/results', BOLTZ_BUDGET_H='9.5',
                  BOLTZ_W='/tmp/boltz_work', BOLTZ_CACHE='/tmp/boltz_cache', BOLTZ_BIN='/tmp/venv/bin/boltz',
                  PATH='/tmp/venv/bin:' + os.environ['PATH'])
if MODE == 'smoke':
    os.environ['BOLTZ_ONLY'] = 'donepezil'
PY = '/tmp/venv/bin/python scripts/boltz_run.py'
print(os.listdir(root))

In [ ]:
sh(PY + ' inputs')

In [ ]:
# 3. set A: robust core + controls (smoke: donepezil only)
sh(PY + ' setA')

In [ ]:
# 4. set B: 40 binders vs 40 decoys (skipped in smoke mode)
if MODE == 'full':
    sh(PY + ' setB')
else:
    print('smoke mode: set B skipped')

In [ ]:
# 5. collect and keep only small outputs
sh(PY + ' collect')
os.makedirs('/kaggle/working/results', exist_ok=True)
for f in glob.glob('results/BOLTZ2_*.csv'):
    shutil.copy(f, '/kaggle/working/results/')
import pandas as pd
print(pd.read_csv('results/BOLTZ2_RESULTS.csv').drop(columns=['smiles']).to_string())
# structures of set A (small): the predicted complexes in mmCIF
os.makedirs('/kaggle/working/structures', exist_ok=True)
for f in glob.glob('/tmp/boltz_work/out/**/*_model_0.cif', recursive=True):
    n = os.path.basename(f)
    if not (n.startswith('act_') or n.startswith('dec_')):
        shutil.copy(f, '/kaggle/working/structures/')
print(os.listdir('/kaggle/working'))
print('NOTEBOOK_COMPLETED_OK')